# Product Pricer — Fine-Tuning a Frontier Model
This notebook demonstrates how to fine-tune a frontier model for product price prediction, following a robust, reproducible, and extensible workflow. 

## 1. Import Required Libraries


In [ ]:
# Imports
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import gradio as gr
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings


# Give this notebook access to the week6 pricer module
sys.path.insert(0, "../../")
from pricer.items import Item
from pricer.evaluator import evaluate

warnings.filterwarnings('ignore')
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if openrouter_api_key and openrouter_api_key.startswith('sk-'):
    print(f"OpenRouter API key found: {openrouter_api_key[:12]}...")
else:
    print("OpenRouter API key not found — check your .env file")

## 2. Load and Inspect Dataset
Load the dataset for fine-tuning. Display basic statistics and a few sample rows to understand the data structure.

In [ ]:
# Load preprocessed dataset using pricer module

print("Loading dataset from HuggingFace (ed-donner/items_lite)...")
train, val, test = Item.from_hub("ed-donner/items_lite")
print(f"Train: {len(train):,} items")
print(f"Val:   {len(val):,} items")
print(f"Test:  {len(test):,} items")
print()
print("Sample item:")
print(train[0])
print()
print("Sample summary:")
print(train[0].summary)

## 3. Load Pre-trained Frontier Model
Load the pre-trained frontier model that will be fine-tuned. Ensure the model is compatible with the dataset and task.

In [ ]:
# Example: Load a pre-trained OpenAI model (or HuggingFace model if using transformers)
BASE_MODEL = "gpt-4.1-nano-2025-04-14"
print(f"Base model: {BASE_MODEL}")
# For OpenAI: from openai import OpenAI

## 4. Configure Fine-Tuning Parameters
Set up hyperparameters such as learning rate, batch size, number of epochs, and optimizer. Consider using a configuration file or dictionary for easy adjustments.

In [ ]:
# Example hyperparameters
config = {
    'learning_rate': 2e-5,
    'batch_size': 4,
    'epochs': 1,
    'seed': 42,
    'max_length': 256,
}
print(json.dumps(config, indent=2))

## 5. Fine-Tune the Model
Train the model on the training set using the specified parameters. Include code for monitoring training progress and saving checkpoints.

In [ ]:
# Example: Prepare data for OpenAI fine-tuning API
import openai  # Added import for OpenAI API
from dotenv import load_dotenv
import os
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in environment. Please add it to your .env file.")
openai.api_key = openai_api_key
def to_jsonl(df, path):
    with open(path, 'w') as f:
        for _, row in df.iterrows():
            messages = [
                {"role": "system", "content": "You estimate prices of products. Reply only with the price, no explanation."},
                {"role": "user", "content": row['prompt']},
                {"role": "assistant", "content": row['response']}
            ]
            f.write(json.dumps({"messages": messages}) + '\n')

import pandas as pd
os.makedirs('jsonl', exist_ok=True)
# Convert train and val lists to DataFrames with prompt/response fields
train_df = pd.DataFrame([{'prompt': item.prompt, 'response': f'Price is ${item.price:.2f}'} for item in train])
val_df = pd.DataFrame([{'prompt': item.prompt, 'response': f'Price is ${item.price:.2f}'} for item in val])
to_jsonl(train_df, 'jsonl/train.jsonl')
to_jsonl(val_df, 'jsonl/val.jsonl')

# Submit fine-tuning job (OpenAI API example)
with open('jsonl/train.jsonl', 'rb') as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
with open('jsonl/val.jsonl', 'rb') as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")

job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model=BASE_MODEL,
    seed=config['seed'],
    hyperparameters={"n_epochs": config['epochs'], "batch_size": config['batch_size']},
    suffix="pricer-demo"
    )
print('Fine-tuning job submitted:', job.id)

## 6. Evaluate Model Performance
Evaluate the fine-tuned model on the validation set. Display metrics such as accuracy, loss, or other relevant scores.

In [ ]:
# Example: Evaluate the fine-tuned model (OpenAI API)
def get_price(response_text):
    s = response_text.replace('$', '').replace(',', '')
    import re
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else 0.0

def evaluate_model(model_name, val_df):
    preds = []
    for _, row in val_df.iterrows():
        response = openai.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You estimate prices of products. Reply only with the price, no explanation."},
                {"role": "user", "content": row['prompt']}
            ],
            max_tokens=7
        )
        price = get_price(response.choices[0].message.content)
        preds.append(price)
    mae = mean_absolute_error(val_df['price'], preds)
    print(f"Validation MAE: ${mae:.2f}")
    return preds

# model_name = job.fine_tuned_model  # Uncomment after job completion
## preds = evaluate_model(model_name, val_df)

## 7. Save Fine-Tuned Model
Save the fine-tuned model and any associated artifacts for future use or deployment.

In [ ]:
# Save model details (for OpenAI, save job/model IDs; for HuggingFace, save model weights)
with open('fine_tuned_model_info.json', 'w') as f:
    json.dump({'job_id': job.id, 'base_model': BASE_MODEL}, f)
print('Model info saved.')

In [ ]:
# Gradio UI for fine-tuned product price prediction (Amazon dataset, pricer module, OpenRouter)
import gradio as gr
import requests

best_model_name = "gpt-4.1-nano-2025-04-14"  # Replace with your actual fine-tuned model name

def predict_price(description):
    if not description.strip():
        return "Please enter a product description."
    headers = {"Authorization": f"Bearer {openrouter_api_key}"}
    payload = {
        "model": best_model_name,
        "messages": [
            {"role": "system", "content": "You estimate prices of products. Reply only with the price, no explanation."},
            {"role": "user", "content": f"Estimate the price of this product.\n\n{description}"},
        ],
        "max_tokens": 7
    }
    response = requests.post("https://openrouter.ai/api/v1/chat/completions", json=payload, headers=headers)
    price = get_price(response.json()['choices'][0]['message']['content'])
    return f"${price:.2f}"

# Show a sample item from the loaded dataset
sample_item = train[0] if train else None
sample_text = f'Title: {sample_item.title}\nCategory: {sample_item.category}\nDescription: {sample_item.summary}' if sample_item else 'Sample unavailable.'

EXAMPLES = [
    sample_text,
    "Title: Wireless Noise-Cancelling Headphones\nCategory: Electronics\nBrand: Sony\nDescription: Over-ear Bluetooth headphones with 30hr battery and active noise cancellation.\nDetails: Foldable design, built-in microphone, USB-C charging.",
    "Title: Cordless Power Drill\nCategory: Tools and Home Improvement\nBrand: DeWalt\nDescription: 20V MAX lithium-ion drill with variable speed and LED work light.\nDetails: 2-speed transmission, 1/2 inch chuck, includes battery and charger.",
    "Title: Stainless Steel Cookware Set\nCategory: Appliances\nBrand: Cuisinart\nDescription: 12-piece tri-ply bonded cookware set with glass lids.\nDetails: Oven safe to 500F, dishwasher safe, induction compatible.",
    "Title: Kids Building Blocks Set\nCategory: Toys and Games\nBrand: Mega Bloks\nDescription: 80-piece large building blocks for toddlers aged 1-5.\nDetails: BPA-free, non-toxic, compatible with Duplo.",
]

with gr.Blocks(title="Product Pricer — Amazon Price Prediction", theme=gr.themes.Soft()) as ui:
    gr.Markdown(f"""
    # Product Pricer — Amazon Price Prediction
    This tool uses a fine-tuned model (via pricer module) to estimate product prices from Amazon product descriptions using OpenRouter API.
    Enter a product description below (or use a sample) to get a price estimate.
    **Sample item from dataset:**
    {sample_text}
    """)

    with gr.Row():
        with gr.Column(scale=2):
            description = gr.Textbox(
                label="Product Description",
                placeholder="Title: ...\nCategory: ...\nBrand: ...\nDescription: ...\nDetails: ...",
                lines=7
            )
            submit = gr.Button("Estimate Price", variant="primary")
            gr.Examples(examples=EXAMPLES, inputs=description)
        with gr.Column(scale=1):
            output = gr.Textbox(label="Estimated Price", lines=2, interactive=False)

    submit.click(fn=predict_price, inputs=description, outputs=output)

ui.launch(inbrowser=True)
